# R2-01 · Datos: traer, cruzar y limpiar — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R2 · *Científico de Datos* · Semana 3 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. **Cruzar** mediciones con su catálogo usando `merge` (y auditar huérfanas).
2. Corregir **tipos**: texto → fecha, y extraer año y mes.
3. Diagnosticar **faltantes reales** y decidir: descartar, imputar o dejar.
4. Revisar **duplicados de llave** y **valores imposibles**.

**Competencia de salida:** construir un dataset analítico limpio, con sus decisiones de limpieza declaradas.

**Dato real:** PM2.5 y meteorología diaria de 14 estaciones SINCA (2020–2025) × catálogo de estaciones (comuna, región, coordenadas).

**Entregable:** que las **4 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# "En vivo o caché": usa los archivos locales; si faltan (ej. Colab), se bajan del repo.
for archivo in ["aire_diario.csv", "estaciones.csv"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{archivo}", archivo)

df = pd.read_csv("aire_diario.csv")          # mediciones: un día por estación
estaciones = pd.read_csv("estaciones.csv")   # catálogo: dónde está cada estación
print("mediciones:", df.shape, "| estaciones:", estaciones.shape)
df.head(3)

## Dos tablas, un dataset analítico

- `df` — las **mediciones**: un día por estación con su PM2.5 (µg/m³), temperatura y humedad.
- `estaciones` — el **catálogo**: comuna, región y coordenadas de cada una de las 14 estaciones.

Un modelo (y una jefatura) necesita las dos cosas juntas: *«¿cuánto PM2.5 hubo, dónde y
cuándo?»*. En este módulo vas a construir ese **dataset analítico** paso a paso — y en el camino
vas a encontrar problemas **reales**, no inventados: estaciones que nunca midieron temperatura,
días sin dato de PM2.5 y una humedad imposible. Para cada uno vas a tomar (y declarar) una decisión.

> 🧭 **Regla de la casa:** limpiar no es «borrar lo feo». Es **diagnosticar, decidir y dejar
> escrito por qué**.

## 1. Cruzar tablas con `merge`

La columna `estacion` existe en ambas tablas: es la **llave**. Con `how="left"` conservas **todas** las mediciones y les pegas la comuna y la región; si alguna estación no estuviera en el catálogo, quedaría con región nula (una fila **huérfana**) y la podrías contar.

### ✍️ Ejercicio 1 — Enriquece cada medición con su comuna y región

In [ ]:
aire = df.merge(estaciones, on="estacion", how="left")
n_huerfanas = int(aire["region"].isna().sum())
print("filas:", len(aire), "| huérfanas:", n_huerfanas)
print(aire[["fecha", "estacion", "comuna", "region", "pm25"]].head(3))

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert "region" in aire.columns and "comuna" in aire.columns
    assert len(aire) == len(df)
    assert n_huerfanas == int(aire["region"].isna().sum())
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "df.merge(estaciones, on='estacion', how='left'); las huérfanas con aire['region'].isna().sum().")
    print("   Detalle:", e)

## 2. Corregir tipos: las fechas

`fecha` llegó como **texto** (`object`): pandas no sabe que «2024-07-01» es un día de invierno. Con `pd.to_datetime` la conviertes en fecha de verdad y puedes extraer el **año** y el **mes** (`.dt.year`, `.dt.month`) — variables clave para un fenómeno tan estacional como el PM2.5.

### ✍️ Ejercicio 2 — Convierte `fecha` a datetime y crea `anio` y `mes`

In [ ]:
aire["fecha"] = pd.to_datetime(aire["fecha"])
aire["anio"] = aire["fecha"].dt.year
aire["mes"] = aire["fecha"].dt.month
print(aire.dtypes[["fecha", "anio", "mes"]])
print("desde", aire["fecha"].min().date(), "hasta", aire["fecha"].max().date())

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert pd.api.types.is_datetime64_any_dtype(aire["fecha"])
    assert set(aire["mes"].unique()) == set(range(1, 13))
    assert aire["anio"].min() == 2020 and aire["anio"].max() == 2025
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "pd.to_datetime(aire['fecha']); luego aire['fecha'].dt.year y aire['fecha'].dt.month.")
    print("   Detalle:", e)

## 3. Faltantes reales: diagnosticar antes de decidir

Hay dos tipos de huecos muy distintos:

- **Meteorología:** algunas estaciones **no tienen sensor** de temperatura — su `temp_media` está vacía el 100 % de los días. Imputar ahí sería **inventar** datos.
- **PM2.5:** días sueltos sin medición (mantenimiento, fallas). Para analizar el PM2.5, esas filas no aportan: se pueden **descartar**, declarándolo.

Calcula la proporción de nulos de `temp_media` por estación, lista las estaciones **sin** meteorología y arma `limpio` sin las filas que no tienen PM2.5.

### ✍️ Ejercicio 3 — Diagnostica los nulos y descarta los días sin PM2.5

In [ ]:
pct_nulos_temp = aire["temp_media"].isna().groupby(aire["estacion"]).mean()
sin_meteo = sorted(pct_nulos_temp[pct_nulos_temp == 1].index)
n_nulos_pm25 = int(aire["pm25"].isna().sum())
limpio = aire.dropna(subset=["pm25"])
print(pct_nulos_temp.round(2).sort_values(ascending=False))
print("estaciones sin meteorología:", sin_meteo)
print("días sin PM2.5:", n_nulos_pm25, "| filas tras descartar:", len(limpio))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert abs(pct_nulos_temp.sum() - aire["temp_media"].isna().groupby(aire["estacion"]).mean().sum()) < 1e-9
    assert len(sin_meteo) == 7 and "Antofagasta" in sin_meteo
    assert n_nulos_pm25 == int(aire["pm25"].isna().sum())
    assert int(limpio["pm25"].isna().sum()) == 0 and len(limpio) == len(aire) - n_nulos_pm25
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "aire['temp_media'].isna().groupby(aire['estacion']).mean(); sin_meteo = sorted(pct_nulos_temp[pct_nulos_temp == 1].index); aire.dropna(subset=['pm25']).")
    print("   Detalle:", e)

## 4. Decisiones de limpieza: duplicados y valores imposibles

Dos chequeos que todo dataset analítico merece:

1. **Duplicados de llave:** en esta tabla la combinación `(fecha, estacion)` debería ser única. Si hubiera repetidos, los promedios quedarían inflados. Cuéntalos con `duplicated(subset=...)`.
2. **Valores imposibles:** una humedad relativa media de **0 %** en un día entero no ocurre en Chile habitado — es un error de registro del sensor. La decisión honesta es convertirla en **faltante** (`np.nan`), no dejarla como si fuera real. (Spoiler: en una estación **todos** sus registros de humedad son 0 % — un sensor que nunca midió.)

### ✍️ Ejercicio 4 — Cuenta duplicados y anula las humedades imposibles

In [ ]:
n_dup = int(limpio.duplicated(subset=["fecha", "estacion"]).sum())
limpio = limpio.copy()
n_hum_cero = int((limpio["humedad_media"] == 0).sum())
limpio.loc[limpio["humedad_media"] == 0, "humedad_media"] = np.nan
print("duplicados (fecha, estacion):", n_dup)
print("humedades 0 % convertidas en NaN:", n_hum_cero)
print("¿en qué estación?", aire.loc[aire["humedad_media"] == 0, "estacion"].unique())

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert n_dup == int(limpio.duplicated(subset=["fecha", "estacion"]).sum())
    assert n_hum_cero == int((aire.dropna(subset=["pm25"])["humedad_media"] == 0).sum()) and n_hum_cero > 0
    assert int((limpio["humedad_media"] == 0).sum()) == 0
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "limpio.duplicated(subset=['fecha','estacion']).sum(); (limpio['humedad_media'] == 0).sum(); limpio.loc[limpio['humedad_media'] == 0, 'humedad_media'] = np.nan.")
    print("   Detalle:", e)

In [ ]:
# (ilustración) El dataset analítico ya responde: PM2.5 promedio por mes y región
tabla = limpio.groupby(["mes", "region"])["pm25"].mean().unstack()
tabla.plot(figsize=(8, 4.5), marker="o")
plt.axhline(50, color="#c0392b", ls="--", lw=1, label="norma diaria (50 µg/m³)")
plt.ylabel("PM2.5 promedio (µg/m³)"); plt.xlabel("mes"); plt.xticks(range(1, 13))
plt.title("El invierno pesa: PM2.5 promedio por mes y región (2020–2025)")
plt.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

## ✍️ Reflexión (no se corrige, pero es lo más importante)

Tomaste tres decisiones: **no imputar** la temperatura de 7 estaciones sin sensor, **descartar**
los días sin PM2.5 y **anular** las humedades de 0 %. Escribe en dos líneas la «nota
metodológica» que acompañaría este dataset en un informe: qué hiciste, por qué, y cuántas filas
afectó cada decisión.

> Un dataset sin su nota metodológica es una caja negra. Con ella, es evidencia.

## Cierre

- **`merge`** une mediciones con su catálogo por una **llave**; `how="left"` deja ver las huérfanas.
- Las **fechas** se convierten con `pd.to_datetime`; de ahí salen año y mes, claves en datos estacionales.
- Los **faltantes** se diagnostican antes de tratarlos: no es lo mismo «no hay sensor» que «falló un día».
- **Duplicados de llave** y **valores imposibles** se revisan siempre — y cada decisión se declara.

> *El 80% del trabajo de datos es dejar el dato listo. En R2-02 vas a fabricar variables
> (features) con SQL sobre este mismo dataset: promedios por mes, el PM2.5 de ayer, medias móviles.*